# 311 Complaints Data Pipeline

The script creates a re-runnable data pipe to extract, clean, and transform 311 data, and generate files required by Leaflet.

In [1]:
import requests
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import os
from shapely.geometry import Point 

# os.getcwd()

In [2]:
"""
from google.colab import drive

# Prompt user to mount Google Drive
mount_drive = input("Do you want to mount Google Drive? (yes/no): ").lower()

if mount_drive == 'yes':
    drive.mount('/content/drive')
    content_folder = '/content/drive/MyDrive/nyc310/'
    print(f"Google Drive mounted. content_folder set to: {content_folder}")
else:
    content_folder = '/content/'
    print(f"Google Drive not mounted. content_folder set to: {content_folder}")
"""
content_folder = "./"

## Load and Clean data

In [3]:
df = pd.read_csv(f'{content_folder}nyc_complaints_all.csv')

# drop na
df = df.dropna(subset=['Latitude', 'Longitude','Created Date', 'Complaint Type'])

# create year_month
df['created_date'] = pd.to_datetime(df['Created Date'])
df['year_month'] = df['created_date'].dt.to_period('M').astype(str)

# merge complaint_types
df['Complaint Type'] = df['Complaint Type'].replace({
    # Basket Complaint
    'Litter Basket Complaint': 'Basket',
    'Overflowing Litter Baskets': 'Basket',
    'Recycling Basket Complaint': 'Basket',
    'Litter Basket / Request': 'Basket',
    'Litter Basket Request': 'Basket',
    'Adopt-A-Basket': 'Basket',
    'Overflowing Recycling Baskets': 'Basket',
    # Sweeping Complaint
    'Street Sweeping Complaint': 'Sweeping',
    'Sweeping/Missed-Inadequate': 'Sweeping',
    'Sweeping/Inadequate': 'Sweeping',
    'Sweeping/Missed': 'Sweeping',
    # Missing Collection
    'Missed Collection': 'Collection',
    'Missed Collection (All Materials)': 'Collection',
    'Seasonal Collection' : 'Collection',
    'Request Xmas Tree Collection': 'Collection',
    'Request Large Bulky Item Collection': 'Collection', 
    'Missing Collection': 'Collection',
    'Collection Truck Noise': 'Collection',
    'Change Collection Schedule': 'Collection',
})

# select columns
df = df[['Unique Key', 'created_date', 'year_month', 'Complaint Type', 'Borough','Incident Zip', 'Latitude', 'Longitude']]

C:\Users\42039\AppData\Local\Temp\ipykernel_24236\1639879703.py:1: DtypeWarning: Columns (13,14,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{content_folder}nyc_complaints_all.csv')


In [4]:
print('=========Complaint Types=========', df["Complaint Type"].unique())

=========Complaint Types========= ['Sweeping' 'Collection' 'Basket']


## From `NYC_census_tract.geojson` get `GEOID`

In [5]:
# 1. Load the 'NYC_census_tract.geojson' file into a GeoDataFrame
nyc_census_tract_geo = gpd.read_file(f'../../public/data/311/NYC_census_tract.geojson')

print("GeoDataFrame 'nyc_census_tract_geo' loaded successfully.")

# 2. Display the first few rows of the GeoDataFrame
#print("\nFirst few rows of nyc_census_tract_geo:")
#print(nyc_census_tract_geo.head())

# 3. Print the information about the GeoDataFrame
print("\nDataFrame info for nyc_census_tract_geo:")
nyc_census_tract_geo.info()

# 4. Print the Coordinate Reference System (CRS) of the GeoDataFrame
print(f"\nCRS of nyc_census_tract_geo: {nyc_census_tract_geo.crs}")

GeoDataFrame 'nyc_census_tract_geo' loaded successfully.

DataFrame info for nyc_census_tract_geo:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2325 entries, 0 to 2324
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   OBJECTID       2325 non-null   int32   
 1   CTLabel        2325 non-null   object  
 2   BoroCode       2325 non-null   object  
 3   BoroName       2325 non-null   object  
 4   CT2020         2325 non-null   object  
 5   BoroCT2020     2325 non-null   object  
 6   CDEligibil     2325 non-null   object  
 7   NTAName        2325 non-null   object  
 8   NTA2020        2325 non-null   object  
 9   CDTA2020       2325 non-null   object  
 10  CDTANAME       2325 non-null   object  
 11  GEOID          2325 non-null   object  
 12  PUMA           2325 non-null   object  
 13  Shape__Area    2325 non-null   float64 
 14  Shape__Length  2325 non-null   float64 
 15  geometry       2

The next step is to convert the `df` DataFrame into a GeoDataFrame using the 'Latitude' and 'Longitude' columns. This involves filtering out rows with missing geographical coordinates, creating shapely Point objects, and setting the appropriate CRS (EPSG:4326) to align with the `nyc_census_tract_geo` GeoDataFrame.



In [6]:
from shapely.geometry import Point

# Filter out rows with missing Latitude or Longitude
df_complaints_geo = df.dropna(subset=['Latitude', 'Longitude']).copy()

# Create point geometries
df_complaints_geo['geometry'] = df_complaints_geo.apply(
    lambda row: Point(row['Longitude'], row['Latitude']), axis=1
)

# Convert to GeoDataFrame and set CRS to EPSG:4326
geodata_complaints = gpd.GeoDataFrame(df_complaints_geo, geometry='geometry', crs='EPSG:4326')

print("GeoDataFrame 'geodata_complaints' created successfully from 'df'.")
print("First few rows of geodata_complaints:")
print(geodata_complaints.head())
print(f"\nShape of geodata_complaints: {geodata_complaints.shape}")
print(f"\nCRS of geodata_complaints: {geodata_complaints.crs}")

GeoDataFrame 'geodata_complaints' created successfully from 'df'.
First few rows of geodata_complaints:
     Unique Key        created_date year_month Complaint Type   Borough  \
180    27025605 2013-12-31 12:27:00    2013-12       Sweeping  BROOKLYN   
181    27028413 2013-12-31 11:13:00    2013-12       Sweeping     BRONX   
182    27027316 2013-12-31 10:27:00    2013-12       Sweeping  BROOKLYN   
183    27026170 2013-12-31 10:05:00    2013-12       Sweeping     BRONX   
184    27016863 2013-12-30 14:19:00    2013-12       Sweeping    QUEENS   

     Incident Zip   Latitude  Longitude                    geometry  
180       11214.0  40.589433 -73.990070  POINT (-73.99007 40.58943)  
181       10472.0  40.828884 -73.866614  POINT (-73.86661 40.82888)  
182       11215.0  40.670480 -73.980245  POINT (-73.98025 40.67048)  
183       10472.0  40.835547 -73.877973  POINT (-73.87797 40.83555)  
184       11369.0  40.759713 -73.863871  POINT (-73.86387 40.75971)  

Shape of geodata_complai

The next step is to spatially join the `geodata_complaints` with `nyc_census_tract_geo` to assign a census tract ID to each complaint. Then, the extracted census tract IDs will be merged back into the main `df` DataFrame based on the 'Unique Key' to fulfill the task's requirement.



In [7]:
# Ensure both GeoDataFrames have the same CRS before spatial joining
# geodata_complaints and nyc_census_tract_geo already have EPSG:4326

# Perform spatial join using 'within' predicate
# Use a left join to keep all complaints from geodata_complaints
complaints_with_census_tract = gpd.sjoin(
    geodata_complaints,
    nyc_census_tract_geo[['GEOID', 'geometry']], # Select only necessary columns from census tracts
    how='left',
    predicate='within'
)

print("Spatial join completed successfully.")
print("First few rows of complaints_with_census_tract (joined with census tract data):")
print(complaints_with_census_tract.head())
print(f"\nShape of complaints_with_census_tract: {complaints_with_census_tract.shape}")

# Prepare the census tract ID for merging back to the original df
census_tract_mapping = complaints_with_census_tract[['Unique Key', 'GEOID']].copy()
census_tract_mapping.rename(columns={'GEOID': 'census_tract_id'}, inplace=True)

# Merge the census_tract_id back into the original df DataFrame
df = df.merge(census_tract_mapping, on='Unique Key', how='left')

print("\nCensus tract IDs merged back into the original 'df' DataFrame.")
print("First few rows of 'df' with the new 'census_tract_id' column:")
print(df[['Unique Key', 'Latitude', 'Longitude', 'census_tract_id']].head())
print(f"\nColumns in 'df' after merge: {df.columns.tolist()}")

Spatial join completed successfully.
First few rows of complaints_with_census_tract (joined with census tract data):
     Unique Key        created_date year_month Complaint Type   Borough  \
180    27025605 2013-12-31 12:27:00    2013-12       Sweeping  BROOKLYN   
181    27028413 2013-12-31 11:13:00    2013-12       Sweeping     BRONX   
182    27027316 2013-12-31 10:27:00    2013-12       Sweeping  BROOKLYN   
183    27026170 2013-12-31 10:05:00    2013-12       Sweeping     BRONX   
184    27016863 2013-12-30 14:19:00    2013-12       Sweeping    QUEENS   

     Incident Zip   Latitude  Longitude                    geometry  \
180       11214.0  40.589433 -73.990070  POINT (-73.99007 40.58943)   
181       10472.0  40.828884 -73.866614  POINT (-73.86661 40.82888)   
182       11215.0  40.670480 -73.980245  POINT (-73.98025 40.67048)   
183       10472.0  40.835547 -73.877973  POINT (-73.87797 40.83555)   
184       11369.0  40.759713 -73.863871  POINT (-73.86387 40.75971)   

     

## From `Community_Districts.csv` get `BoroCD`

In [8]:
df_community_districts = pd.read_csv(f'{content_folder}Community_Districts.csv')

In [9]:
from shapely import wkt

# 1. Convert 'the_geom' column to geometry objects
df_community_districts['geometry'] = gpd.GeoSeries.from_wkt(df_community_districts['the_geom'])

# 2. Create a new GeoDataFrame named geodata_community_districts
geodata_community_districts = gpd.GeoDataFrame(df_community_districts, geometry='geometry')

# 3. Set the Coordinate Reference System (CRS) to EPSG:4326
geodata_community_districts.crs = 'EPSG:4326'

print("GeoDataFrame 'geodata_community_districts' created successfully.")
print(f"\nShape of geodata_community_districts: {geodata_community_districts.shape}")
print(f"\nCRS of geodata_community_districts: {geodata_community_districts.crs}")

GeoDataFrame 'geodata_community_districts' created successfully.

Shape of geodata_community_districts: (71, 5)

CRS of geodata_community_districts: EPSG:4326


In [10]:
complaints_with_community_districts = gpd.sjoin(
    geodata_complaints,
    geodata_community_districts[['BoroCd', 'geometry']],
    how='left',
    predicate='within'
)

print("Spatial join completed successfully, assigning 'BoroCd' to complaints.")


Spatial join completed successfully, assigning 'BoroCd' to complaints.


In [11]:
community_district_mapping = complaints_with_community_districts[['Unique Key', 'BoroCd']].copy()

df = df.merge(community_district_mapping, on='Unique Key', how='left')

df.dropna(subset=['BoroCd'], inplace=True)
df['BoroCd'] = df.BoroCd.astype(int)

print("\nBoroCD merged back into the original 'df' DataFrame.")
df.head()


BoroCD merged back into the original 'df' DataFrame.


,Unique Key,created_date,year_month,Complaint Type,Borough,Incident Zip,Latitude,Longitude,census_tract_id,BoroCd
0,27025605,2013-12-31 12:27:00,2013-12,Sweeping,BROOKLYN,11214.0,40.589433,-73.990070,36047031401,313
1,27028413,2013-12-31 11:13:00,2013-12,Sweeping,BRONX,10472.0,40.828884,-73.866614,36005007000,209
2,27027316,2013-12-31 10:27:00,2013-12,Sweeping,BROOKLYN,11215.0,40.670480,-73.980245,36047015500,306
3,27026170,2013-12-31 10:05:00,2013-12,Sweeping,BRONX,10472.0,40.835547,-73.877973,36005006200,209
4,27016863,2013-12-30 14:19:00,2013-12,Sweeping,QUEENS,11369.0,40.759713,-73.863871,36081036500,403


In [12]:
df.to_csv(f'{content_folder}nyc_complaints_cleaned.csv', index=False)

## Aggregate on census_tract_id

In [13]:
# Aggregate on year_month, complaint_type, and census_tract_id
aggregated_df = df.groupby(['year_month', 'Complaint Type', 'census_tract_id']).size().reset_index(name='complaint_count')
aggregated_df.to_csv(content_folder + 'aggregated_complaint_census_tract.csv', index=False)
aggregated_df.to_json(content_folder + 'aggregated_complaint_census_tract.json', orient='records', indent=4)
print("DataFrame saved to 'aggregated_complaint_census_tract.json'")

DataFrame saved to 'aggregated_complaint_census_tract.json'


### Create optimized JSON

In [14]:
sum_df = aggregated_df.groupby(['year_month', 'census_tract_id'])['complaint_count'].sum().reset_index()
sum_df['Complaint Type'] = 'All'
aggregated_df = pd.concat([aggregated_df, sum_df], ignore_index=True)


In [15]:
# Create Nested Dictionary Structure: Complaint Type -> year_month -> census_tract_id -> complaint_count
optimized_data = {}

for index, row in aggregated_df.iterrows():
    complaint_type = row['Complaint Type']
    year_month = row['year_month']
    census_tract_id = row['census_tract_id']
    complaint_count = row['complaint_count']

    if complaint_type not in optimized_data:
        optimized_data[complaint_type] = {}

    if year_month not in optimized_data[complaint_type]:
        optimized_data[complaint_type][year_month] = {}

    optimized_data[complaint_type][year_month][census_tract_id] = complaint_count

print("Optimized nested dictionary created successfully.")


Optimized nested dictionary created successfully.


In [16]:
# save JSON file
import json

output_json_path = '../../public/data/311/optimized_complaint_census_tract.json'

with open(output_json_path, 'w') as f:
    json.dump(optimized_data, f, indent=None) # Set indent to None for compact JSON

print(f"Optimized nested dictionary saved to '{output_json_path}' successfully.")

Optimized nested dictionary saved to '../../public/data/311/optimized_complaint_census_tract.json' successfully.


## Aggregate on community district

In [17]:
# Aggregate on year_month, complaint_type, and BoroCd
aggregated_df = df.groupby(['year_month', 'Complaint Type', 'BoroCd']).size().reset_index(name='complaint_count')
aggregated_df.to_csv(content_folder + 'aggregated_complaint_community_district.csv', index=False)

In [18]:
','.join(sorted(aggregated_df.BoroCd.astype(str).unique()))

'101,102,103,104,105,106,107,108,109,110,111,112,164,201,202,203,204,205,206,207,208,209,210,211,212,226,227,228,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,355,356,401,402,403,404,405,406,407,408,409,410,411,412,413,414,480,481,482,484,501,502,503,595'

## Create file that will not include "Request Large Bulky Item Collection" category


In [19]:
# Paths
csv_path = "nyc_complaints_all.csv"
geojson_path = "../../public/data/311/NYC_census_tract.geojson"
output_json_path = "../../public/data/311/optimized_complaint_census_tract_no_bulky.json"

# Load 311 data
df = pd.read_csv(csv_path)

# Remove bulky item requests
EXCLUDED_TYPES = ["Request Large Bulky Item Collection"]
df = df[~df["Complaint Type"].isin(EXCLUDED_TYPES)]

# Drop missing values
df = df.dropna(subset=["Latitude", "Longitude", "Created Date", "Complaint Type"])

# Parse dates
df["created_date"] = pd.to_datetime(df["Created Date"])
df["year_month"] = df["created_date"].dt.to_period("M").astype(str)

# Simplify complaint types
merge_types = {
    "Litter Basket Complaint": "Basket",
    "Overflowing Litter Baskets": "Basket",
    "Recycling Basket Complaint": "Basket",
    "Litter Basket / Request": "Basket",
    "Litter Basket Request": "Basket",
    "Adopt-A-Basket": "Basket",
    "Overflowing Recycling Baskets": "Basket",

    "Street Sweeping Complaint": "Sweeping",
    "Sweeping/Missed-Inadequate": "Sweeping",
    "Sweeping/Inadequate": "Sweeping",
    "Sweeping/Missed": "Sweeping",

    "Missed Collection": "Collection",
    "Missed Collection (All Materials)": "Collection",
    "Seasonal Collection": "Collection",
    "Request Xmas Tree Collection": "Collection",
    "Missing Collection": "Collection",
    "Collection Truck Noise": "Collection",
    "Change Collection Schedule": "Collection",
}

df["Complaint Type"] = df["Complaint Type"].replace(merge_types)

# Keep needed columns
df = df[[
    "Unique Key", "created_date", "year_month",
    "Complaint Type", "Borough", "Incident Zip",
    "Latitude", "Longitude"
]]

# Load census tract GeoJSON
tracts = gpd.read_file(geojson_path)

# Convert complaints to GeoDataFrame
df["geometry"] = df.apply(lambda r: Point(r["Longitude"], r["Latitude"]), axis=1)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# Spatial Join (find census tract for each complaint)
joined = gpd.sjoin(
    gdf,
    tracts[["GEOID", "geometry"]],
    how="left",
    predicate="within"
)

# Add GEOID back to df
df["census_tract_id"] = joined["GEOID"]

# Aggregate counts
agg = df.groupby(
    ["year_month", "Complaint Type", "census_tract_id"]
).size().reset_index(name="complaint_count")

# Also create "All" combined category
sum_df = agg.groupby(["year_month", "census_tract_id"])["complaint_count"] \
            .sum().reset_index()
sum_df["Complaint Type"] = "All"

agg = pd.concat([agg, sum_df], ignore_index=True)

# Build optimized nested JSON
optimized = {}

for _, row in agg.iterrows():
    ctype = row["Complaint Type"]
    ym = row["year_month"]
    tract = row["census_tract_id"]
    count = row["complaint_count"]

    optimized.setdefault(ctype, {})
    optimized[ctype].setdefault(ym, {})
    optimized[ctype][ym][tract] = count

# Save JSON (compact)
with open(output_json_path, "w") as f:
    json.dump(optimized, f, indent=None)

print("✔ Saved:", output_json_path)

C:\Users\42039\AppData\Local\Temp\ipykernel_24236\3133794021.py:7: DtypeWarning: Columns (13,14,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


✔ Saved: ../../public/data/311/optimized_complaint_census_tract_no_bulky.json


## Create aggregated data from analysis

In [20]:
# Paths
csv_path = "nyc_complaints_all.csv"
output_csv = "nyc_complaints_by_borough.csv"

# Load 311 data
df = pd.read_csv(csv_path)

# Remove bulky item requests
EXCLUDED_TYPES = ["Request Large Bulky Item Collection"]
df = df[~df["Complaint Type"].isin(EXCLUDED_TYPES)]

# Drop missing values
df = df.dropna(subset=["Latitude", "Longitude", "Created Date", "Complaint Type"])

# Parse dates
df["created_date"] = pd.to_datetime(df["Created Date"])
df["year_month"] = df["created_date"].dt.to_period("M").astype(str)
df["year"] = df['created_date'].dt.year
df["month"] = df['created_date'].dt.month

# Simplify complaint types
merge_types = {
    "Litter Basket Complaint": "Basket",
    "Overflowing Litter Baskets": "Basket",
    "Recycling Basket Complaint": "Basket",
    "Litter Basket / Request": "Basket",
    "Litter Basket Request": "Basket",
    "Adopt-A-Basket": "Basket",
    "Overflowing Recycling Baskets": "Basket",

    "Street Sweeping Complaint": "Sweeping",
    "Sweeping/Missed-Inadequate": "Sweeping",
    "Sweeping/Inadequate": "Sweeping",
    "Sweeping/Missed": "Sweeping",

    "Missed Collection": "Collection",
    "Missed Collection (All Materials)": "Collection",
    "Seasonal Collection": "Collection",
    "Request Xmas Tree Collection": "Collection",
    "Missing Collection": "Collection",
    "Collection Truck Noise": "Collection",
    "Change Collection Schedule": "Collection",
}

df["Complaint Type"] = df["Complaint Type"].replace(merge_types)

# aggregate
df_aggregate = df.groupby(['year', 'month', 'Complaint Type', 'Borough']).size().reset_index(name='count')

df_aggregate.to_csv(output_csv, index=False)

print("✔ Saved:", output_csv)

C:\Users\42039\AppData\Local\Temp\ipykernel_24236\2407716461.py:6: DtypeWarning: Columns (13,14,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


✔ Saved: nyc_complaints_by_borough.csv
